# OFF Ontology Data Cleaning (No Dedup)
Applies the same cleaning pipeline as `eda2.ipynb` to `off_data_ontology.csv`:
- Strip & lowercase
- Remove entries with `item_name` > 5 words
- Remove measurement keywords
- Remove entries containing digits
- Normalise: unicode → ASCII, special chars, extra whitespace

## Step 1 — Load Data

In [ ]:
import pandas as pd
import re
import unicodedata

df = pd.read_csv('off_data_ontology.csv', low_memory=False)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(df.columns.tolist())
df.head(3)

## Step 2 — Ensure `item_name` is a string & strip whitespace

In [ ]:
# Cast to string and strip
df['item_name'] = df['item_name'].astype(str).str.strip()

# Drop rows where item_name ended up empty or 'nan'
before = len(df)
df = df[df['item_name'].str.lower() != 'nan']
df = df[df['item_name'] != '']
print(f"Dropped {before - len(df):,} null/empty item_name rows → {len(df):,} remaining")

## Step 3 — Lowercase & normalise hyphens

In [ ]:
df['item_name'] = df['item_name'].str.lower().str.replace('-', ' ', regex=False)
print(df['item_name'].head(5).tolist())

## Step 4 — Filter: keep `item_name` ≤ 5 words

In [ ]:
df['item_name_word_count'] = df['item_name'].str.split().apply(len)
before = len(df)
df = df[df['item_name_word_count'] <= 5]
print(f"Dropped {before - len(df):,} rows with >5-word item names → {len(df):,} remaining")
print(df['item_name_word_count'].value_counts().sort_index())

## Step 5 — Remove measurement keywords

In [ ]:
food_measurements = [
    # Weight / mass
    "mg", "milligram", "milligrams",
    "g", "gram", "grams",
    "kg", "kilogram", "kilograms",
    "oz", "ounce", "ounces",
    "lb", "lbs", "pound", "pounds",
    # Volume
    "ml", "milliliter", "milliliters", "millilitre", "millilitres",
    "cl", "centiliter", "centiliters", "centilitre", "centilitres",
    "l", "liter", "liters", "litre", "litres",
    "tsp", "teaspoon", "teaspoons",
    "tbsp", "tablespoon", "tablespoons",
    "fl oz", "fluid ounce", "fluid ounces",
    "cup", "cups",
    "pint", "pints",
    "quart", "quarts",
    "gallon", "gallons",
    "dash", "dashes",
    "pinch", "pinches",
    "splash", "splashes",
    "drop", "drops",
    # Count / piece-based
    "piece", "pieces",
    "pc", "pcs",
    "unit", "units",
    "item", "items",
    "whole", "halves", "half",
    "quarter", "quarters",
    "slice", "slices",
    "stick", "sticks",
    "cube", "cubes",
    "chunk", "chunks",
    "wedge", "wedges",
    "strip", "strips",
    "ring", "rings",
    "clove", "cloves",
    "leaf", "leaves",
    "sprig", "sprigs",
    "stalk", "stalks",
    "stem", "stems",
    "head", "heads",
    "bunch", "bunches",
    "bulb", "bulbs",
    "ear", "ears",
    "kernel", "kernels",
    "pod", "pods",
    "bean", "beans",
    "egg", "eggs",
    "fillet", "fillets",
    "breast", "breasts",
    "thigh", "thighs",
    "drumstick", "drumsticks",
    "leg", "legs",
    "wing", "wings",
    # Serving style
    "serving", "servings",
    "portion", "portions",
    "helping", "helpings",
    "plate", "plates",
    "bowl", "bowls",
    "dish", "dishes",
    "tray", "trays",
    # Packaging / container-based
    "pack", "packs",
    "packet", "packets",
    "package", "packages",
    "bag", "bags",
    "box", "boxes",
    "carton", "cartons",
    "can", "cans",
    "tin", "tins",
    "jar", "jars",
    "bottle", "bottles",
    "tube", "tubes",
    "sachet", "sachets",
    "wrapper", "wrappers",
    "container", "containers",
    "cupful", "cupfuls",
    # Bakery / produce / retail
    "loaf", "loaves",
    "roll", "rolls",
    "bun", "buns",
    "patty", "patties",
    "link", "links",
    "sausage", "sausages",
    "ball", "balls",
    "bar", "bars",
    "block", "blocks",
    # Recipe amounts
    "handful", "handfuls",
    "fistful", "fistfuls",
    "scoop", "scoops",
    "ladle", "ladles",
    "spoonful", "spoonfuls",
    "heaped teaspoon", "heaped teaspoons",
    "heaped tablespoon", "heaped tablespoons",
    "level teaspoon", "level teaspoons",
    "level tablespoon", "level tablespoons",
    "to taste",
]

pattern = r'\b(?:' + '|'.join(re.escape(m) for m in food_measurements) + r')\b'
before = len(df)
df = df[~df['item_name'].str.contains(pattern, regex=True)]
print(f"Dropped {before - len(df):,} rows containing measurement keywords → {len(df):,} remaining")

## Step 6 — Remove entries containing digits

In [ ]:
before = len(df)
df = df[~df['item_name'].str.contains(r'\d', regex=True)]
print(f"Dropped {before - len(df):,} rows with digits in item_name → {len(df):,} remaining")

## Step 7 — Normalise: Unicode → ASCII, remove special chars, collapse whitespace

In [ ]:
def normalize_item_name(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = s.replace("&", " and ")
    s = re.sub(r"[^\w\s]", " ", s)          # remove special chars
    s = re.sub(r"\s+", " ", s).strip()      # collapse whitespace
    return s

df['item_name'] = df['item_name'].apply(normalize_item_name)

# Drop any rows that became empty after normalisation
before = len(df)
df = df[df['item_name'] != '']
print(f"Dropped {before - len(df):,} rows that became empty after normalisation → {len(df):,} remaining")
print(df['item_name'].head(10).tolist())

## Step 8 — Summary & Save

In [ ]:
print("=== Final Dataset ===")
print(f"Rows : {len(df):,}")
print(f"Cols : {df.columns.tolist()}")
print(f"Unique item_names: {df['item_name'].nunique():,}")
print()
print(df.dtypes)
df.head(5)

In [ ]:
df.to_csv('off_data_ontology_clean.csv', index=False)
print("Saved → off_data_ontology_clean.csv")